In [1]:
import os
import numpy as np
from keras.models import load_model
from pkg_resources import resource_filename
from spliceai.utils import one_hot_encode
import tensorflow as tf

# Silence TensorFlow output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# Paths
base_path = '/home/mikej10/advbfx'
extracted_dir = os.path.join(base_path, 'intronretention', 'extracted_regions')
output_dir = os.path.join(base_path, 'intronretention')

# FASTA files to process
fasta_files = [
    os.path.join(extracted_dir, '100way52ntSRRMI12.fa'),
    os.path.join(extracted_dir, 'spliceinhnrnph1msa.fa')
]

# Load models
context = 10000
paths = ('models/spliceai{}.h5'.format(x) for x in range(1, 6))
models = [load_model(resource_filename('spliceai', x), compile=False) for x in paths]
predict_fn = [model.predict for model in models]

print("Models loaded successfully")


2025-12-17 11:51:54.758961: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-17 11:51:56.101261: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-17 11:51:56.109489: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-17 11:51:59.681831: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/tmp/ipykernel_123881/2066716930.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_filename
2025-12-17 11:52:07.677125: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_exec

Models loaded successfully


In [3]:
# Function to process a single FASTA file
def process_fasta(fasta_path):
    """
    Process a FASTA file and return predictions for each sequence.
    Returns a list of tuples: (header, max_donor_prob, max_donor_pos)
    Handles multi-line sequences correctly.
    """
    results = []
    
    with open(fasta_path) as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # Check if this is a header line
        if line.startswith('>'):
            header = line[1:]  # Remove '>'
            sequence_parts = []
            
            # Read sequence lines until next header or end of file
            i += 1
            while i < len(lines) and not lines[i].strip().startswith('>'):
                seq_line = lines[i].strip()
                if seq_line:  # Skip empty lines
                    sequence_parts.append(seq_line)
                i += 1
            
            # Combine sequence parts
            sequence = ''.join(sequence_parts).upper()
            
            # Remove gaps/dashes and asterisks from sequence for prediction
            sequence = sequence.replace('-', '').replace('*', '')
            
            # Skip empty sequences
            if len(sequence) == 0:
                continue
            
            # Pad with context
            padded_seq = 'N' * (context // 2) + sequence + 'N' * (context // 2)
            x = one_hot_encode(padded_seq)[None, :]
            
            # Run predictions (ensemble of 5 models)
            y = np.mean([fn(x, verbose=0) for fn in predict_fn], axis=0)
            scores = y[0]  # shape (len + context, 4)
            
            # Extract donor probabilities for the actual sequence (trimming context)
            if len(sequence) >= context:
                start = context // 2
                end = start + len(sequence)
                donor_probs = scores[start:end, 2]
            else:
                donor_probs = scores[:len(sequence), 2]
            
            # Find max donor probability and its position (1-based)
            max_donor_prob = np.max(donor_probs)
            max_donor_pos = np.argmax(donor_probs) + 1  # 1-based position
            
            results.append((header, max_donor_prob, max_donor_pos))
            
            print(f"Processed {header}: max_donor={max_donor_prob:.6f} at position {max_donor_pos}")
        else:
            i += 1
    
    return results


In [4]:
# Process both FASTA files
all_results = []

for fasta_path in fasta_files:
    print(f"\n=== Processing {os.path.basename(fasta_path)} ===")
    results = process_fasta(fasta_path)
    all_results.extend(results)

print(f"\nTotal sequences processed: {len(all_results)}")



=== Processing 100way52ntSRRMI12.fa ===
Processed  hg38: max_donor=0.115773 at position 9
Processed  panTro4: max_donor=0.115773 at position 9
Processed  gorGor3: max_donor=0.115773 at position 9
Processed  ponAbe2: max_donor=0.115773 at position 9
Processed  nomLeu3: max_donor=0.115773 at position 9
Processed  rheMac3: max_donor=0.115773 at position 9
Processed  macFas5: max_donor=0.115773 at position 9
Processed  papAnu2: max_donor=0.115773 at position 9
Processed  chlSab2: max_donor=0.095854 at position 9
Processed  calJac3: max_donor=0.115773 at position 9
Processed  saiBol1: max_donor=0.115773 at position 9
Processed  otoGar3: max_donor=0.127355 at position 9
Processed  tupChi1: max_donor=0.181971 at position 27
Processed  speTri2: max_donor=0.121570 at position 25
Processed  jacJac1: max_donor=0.120922 at position 9
Processed  micOch1: max_donor=0.109458 at position 9
Processed  criGri1: max_donor=0.109458 at position 9
Processed  mesAur1: max_donor=0.109458 at position 9
Proces

In [5]:
# Write results to TSV
output_file = os.path.join(output_dir, 'spliceai_predictions.tsv')

with open(output_file, 'w') as f:
    # Write header
    f.write("genome_name\tmax_donor_probability\tnucleotide_position\n")
    
    # Write data
    for header, max_donor_prob, max_donor_pos in all_results:
        f.write(f"{header}\t{max_donor_prob:.6f}\t{max_donor_pos}\n")

print(f"\nResults written to: {output_file}")
print(f"Total predictions: {len(all_results)}")



Results written to: /home/mikej10/advbfx/intronretention/spliceai_predictions.tsv
Total predictions: 144
